In [1]:
from capstone.data_cleaning import load_full_dataframe
from capstone.helper_functions import setup_logger, load_model_config
from capstone.logistic_regression import train_model
from capstone.setup_project import download_ces_data, download_state_data, install_ncsl_classification

In [2]:
logger = setup_logger()
logger.info("🟢 Starting main capstone project.")

2026-08-14 20:31:37 | INFO     | capstone | 🟢 Starting main capstone project.


In [3]:
cd "/Users/sahanasundar/Desktop/Capstone/capstone"

/Users/sahanasundar/Desktop/Capstone/capstone


In [4]:
config = load_model_config()

# Get the dataframe
df = load_full_dataframe(config)

2026-08-14 20:31:41 | INFO     | capstone | 🟢 Successfully loaded the CES data.
2026-08-14 20:31:41 | INFO     | capstone | 🟢 Beginning cleaning of CES data
2026-08-14 20:31:41 | INFO     | capstone | 🟢 Successfully loaded the FIPS data.
2026-08-14 20:31:41 | INFO     | capstone | 🟢 Successfully loaded the NCSL data.


### Testing No Outreach Numbers for Cleaned Dataframe

In [ ]:
outreach_columns = ["In person", "Phone call", "Email or text message", "Letter or postcard"]
for column in outreach_columns:
    no_outreach = df[(df[column] == 'No')]
    no_outreach_no_vote = df[(df[column] == 'No') & (df['Voted'] == 0)]
    print(column, "\n",  "Number of Respondents who Didn't Vote:", no_outreach_no_vote.shape[0], "\n", "Number of Respondents: ", no_outreach.shape[0])

In [5]:
frame = df.copy()

In [6]:
frame["Contacted"] = (
        frame[
            [
                "In person",
                "Phone call",
                "Email or text message",
                "Letter or postcard",
            ]
        ]
        .eq("Yes")  # type: ignore
        .any(axis=1)
        .map({True: "Contacted", False: "Not Contacted"})
    )

In [20]:
demographic_columns = ["Education", "Race", "Gender"]
for column in demographic_columns:
    print(frame[column].value_counts())

Education
4 year college degree            10603
Some college, no degree (yet)     9414
High school graduate              7599
Postgraduate degree               6583
2 year college degree             4840
No HS degree                       761
Name: count, dtype: int64
Race
White                29727
Black                 4124
Hispanic              2649
Two or more races     1202
Asian                  946
Other                  736
Native American        325
Middle Eastern          91
Name: count, dtype: int64
Gender
Woman         20379
Man           19136
Non-binary      221
Other            64
Name: count, dtype: int64


In [10]:
frame[frame["Contacted"] == "Not Contacted"]

,Education,Race,Gender,Age,In person,Phone call,Email or text message,Letter or postcard,State FIPS Code,Voted,State Name,State Code,NCSL Classification,Contacted
0,High school graduate,Black,Woman,46,No,No,No,No,42,1,Pennsylvania,PA,No Document Required to Vote,Not Contacted
3,High school graduate,White,Woman,23,No,No,No,No,6,1,California,CA,No Document Required to Vote,Not Contacted
7,2 year college degree,Two or more races,Man,56,No,No,No,No,41,1,Oregon,OR,No Document Required to Vote,Not Contacted
8,"Some college, no degree (yet)",Black,Woman,59,No,No,No,No,34,0,New Jersey,NJ,No Document Required to Vote,Not Contacted
9,"Some college, no degree (yet)",Other,Woman,47,No,No,No,No,26,1,Michigan,MI,"Non-Strict, Photo ID",Not Contacted
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39792,High school graduate,White,Woman,21,No,No,No,No,48,1,Texas,TX,"Non-Strict, Photo ID",Not Contacted
39793,High school graduate,White,Woman,44,No,No,No,No,12,0,Florida,FL,"Non-Strict, Photo ID",Not Contacted
39794,No HS degree,White,Man,18,No,No,No,No,47,1,Tennessee,TN,"Strict, Photo ID",Not Contacted
39797,High school graduate,White,Woman,69,No,No,No,No,32,1,Nevada,NV,No Document Required to Vote,Not Contacted


In [7]:
frame[(frame["In person"] == "No") & (frame["Phone call"] == "No") & (frame["Email or text message"] == "No") & (frame["Letter or postcard"] == "No")]

,Education,Race,Gender,Age,In person,Phone call,Email or text message,Letter or postcard,State FIPS Code,Voted,State Name,State Code,NCSL Classification,Contacted
0,High school graduate,Black,Woman,46,No,No,No,No,42,1,Pennsylvania,PA,No Document Required to Vote,Not Contacted
3,High school graduate,White,Woman,23,No,No,No,No,6,1,California,CA,No Document Required to Vote,Not Contacted
7,2 year college degree,Two or more races,Man,56,No,No,No,No,41,1,Oregon,OR,No Document Required to Vote,Not Contacted
8,"Some college, no degree (yet)",Black,Woman,59,No,No,No,No,34,0,New Jersey,NJ,No Document Required to Vote,Not Contacted
9,"Some college, no degree (yet)",Other,Woman,47,No,No,No,No,26,1,Michigan,MI,"Non-Strict, Photo ID",Not Contacted
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39792,High school graduate,White,Woman,21,No,No,No,No,48,1,Texas,TX,"Non-Strict, Photo ID",Not Contacted
39793,High school graduate,White,Woman,44,No,No,No,No,12,0,Florida,FL,"Non-Strict, Photo ID",Not Contacted
39794,No HS degree,White,Man,18,No,No,No,No,47,1,Tennessee,TN,"Strict, Photo ID",Not Contacted
39797,High school graduate,White,Woman,69,No,No,No,No,32,1,Nevada,NV,No Document Required to Vote,Not Contacted


In [ ]:
frame[frame['Contacted'] == 'Not Contacted']

### Testing No Outreach Numbers in Original Dataframe

In [12]:
import pandas as pd

In [13]:
orig_df = pd.read_csv('/Users/sahanasundar/Desktop/Capstone/capstone/capstone/data/CCES24_Common_OUTPUT_vv_topost_final.csv')

In [ ]:
# orig_outreach_columns = ["CC24_431b_1", "CC24_431b_2", "CC24_431b_3", "CC24_431b_4"]
# for column in outreach_columns:
#     no_outreach = df[(df[column].isna())]
#     no_outreach_no_vote = df[(df[column].isna()) & (df["TS_g2024"] == '0')]
#     print(column, "\n",  "Number of Respondents who Didn't Vote:", no_outreach_no_vote.shape[0], "\n", "Number of Respondents: ", no_outreach.shape[0])


In [14]:
orig_df["CC24_431a"].value_counts()

CC24_431a
1.0    29719
2.0    19712
Name: count, dtype: int64

In [15]:
29719 + 19712

49431

In [ ]:
str_test = orig_df["CC24_431b_1"].astype(str)

In [ ]:
str_test[str_test.isna()]

In [11]:
orig_df["CC24_431b_1"].isna().sum()

NameError: name 'orig_df' is not defined